# Sample market database

Load the `sjd` module, create a new SQLite database, and fill it with the dummy pillars in `python/markets/sample.py`.

Build the module first (stages `sjd.pyd` into `python/sjd_runtime`):

`cmake --build --preset debug --target sjd_python`

Shut down the notebook kernel before rebuilding so Windows can overwrite `sjd.pyd`.

The database is `python/markets/sample.db`. Running this notebook again replaces that file.

In [1]:
import sqlite3
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _notebooks = _candidate / "python" / "notebooks"
    if (_notebooks / "sjd_setup.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
    if (_candidate / "sjd_setup.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find python/notebooks/sjd_setup.py from " + str(_cwd))

import sjd_setup

sjd = sjd_setup.load_sjd()
print(sjd.__file__)

E:\Code\sjd\python\sjd_runtime\sjd.pyd


## Create the database

`load_sample` creates the tables, copies the quotes in `python/markets/catalogue.toml` into the key tables, and inserts the 16 September 2026 anchor date. The pillar tables have foreign keys to those key tables, so a row for a quote the catalogue does not store is rejected. XAU ZAR and XPT ZAR forwards are not stored; those quotes are built later from the USD metal curve and USDZAR.

In [2]:
python_dir = sjd_setup.repo_root() / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

from markets.sample import SAMPLE_ANCHOR, load_sample

database_path = python_dir / "markets" / "sample.db"
try:
    connection.close()
except NameError:
    pass
if database_path.exists():
    database_path.unlink()

connection = sqlite3.connect(database_path)
anchor = load_sample(connection)
print(anchor.isoformat())
print(database_path)

2026-09-16
E:\Code\sjd\python\markets\sample.db


## What was stored

Gold spot, one GOFO pillar, the USD discount factor one year out, and how many volatility points each quote has.

In [3]:
anchor_text = SAMPLE_ANCHOR.isoformat()
spot = connection.execute(
    "SELECT value FROM spot WHERE anchor_date = ? AND market = 'XAU' AND currency = 'USD'",
    (anchor_text,),
).fetchone()[0]
gofo = connection.execute(
    """
    SELECT value FROM forward_pillar
    WHERE anchor_date = ? AND market = 'XAU' AND currency = 'USD' AND pillar_date = '2026-12-16'
    """,
    (anchor_text,),
).fetchone()[0]
usd_discount = connection.execute(
    """
    SELECT discount_factor FROM discount_pillar
    WHERE anchor_date = ? AND currency = 'USD' AND pillar_date = '2027-09-16'
    """,
    (anchor_text,),
).fetchone()[0]
volatility_counts = connection.execute(
    """
    SELECT market, currency, COUNT(*)
    FROM volatility_pillar
    WHERE anchor_date = ?
    GROUP BY market, currency
    ORDER BY market, currency
    """,
    (anchor_text,),
).fetchall()

print("anchor", anchor_text)
print("XAU USD spot", spot)
print("XAU USD GOFO 2026-12-16", gofo)
print("USD discount 2027-09-16", usd_discount)
for market, currency, count in volatility_counts:
    print(f"{market} {currency} volatility points", count)

connection.close()

anchor 2026-09-16
XAU USD spot 2400.0
XAU USD GOFO 2026-12-16 0.008
USD discount 2027-09-16 0.9607894391523232
DATED_BRENT USD volatility points 18
USDZAR ZAR volatility points 25
XAU USD volatility points 25
XAU ZAR volatility points 25
XPT USD volatility points 25
XPT ZAR volatility points 25
